# Chapter 10 – Worked examples

This notebook runs the examples of [Chapter 10](../index.md) live. The code lives in the modules next to it
(`per_unit.py`, `three_phase.py`); the cells only call them, so a change in a module is a change here too.

Run it from this folder (the kernel's working directory must be `src/`), or open it in Jupyter Lab from the
repository.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import per_unit
import three_phase

## Per-unit bases

$Z_{\text{base}} = V_{\text{base}}^2 / S_{\text{base}}$, $I_{\text{base}} = S_{\text{base}} / (\sqrt{3}\, V_{\text{base}})$ and the
change of base $z_{\text{new}} = z_{\text{old}} (V_{\text{old}}/V_{\text{new}})^2 (S_{\text{new}}/S_{\text{old}})$: the three
bullets of the worked example.

In [ ]:
print(f'380 kV line rated 1000 A: {per_unit.p_limit_mw(1000.0, 380.0):.0f} MW at unity power factor')
z_base, i_base = per_unit.z_base(220.0), per_unit.i_base(220.0)
print(f'220 kV, 100 MVA: Z_base = {z_base:.0f} ohm, 40 ohm = {40 / z_base:.4f} pu, I_base = {i_base:.0f} A')
print(f'0.10 pu on 50 MVA = {per_unit.change_base(0.10, 1.0, 50.0, 1.0, 100.0):.2f} pu on 100 MVA')

## The per-unit branch table of the lab

`branch_table` adds $Z_{\text{base}}$, $r_{\text{pu}}$, $x_{\text{pu}}$ and the MW limit to the branch list, the way a
power-flow program prepares its data.

In [ ]:
per_unit.branch_table()

## Balanced phasors and the operator $a$

`balanced_set` builds $(V, a^2 V, a V)$; the line voltage $V_{ab} = V_a - V_b$ has magnitude $\sqrt{3}$ and leads
$V_a$ by $30^\circ$.

In [ ]:
v = three_phase.balanced_set()
v_ll = three_phase.line_voltages(v)
print('phase phasors', np.round(v, 4))
print('|V_ab| =', round(abs(v_ll[0]), 4), ' arg(V_ab) =', round(math.degrees(np.angle(v_ll[0])), 1), 'deg')
for name, z in zip('abc', v, strict=True):
    plt.arrow(0, 0, z.real, z.imag, head_width=0.05, length_includes_head=True, label=f'V_{name}')
plt.arrow(v[1].real, v[1].imag, v_ll[0].real, v_ll[0].imag, head_width=0.05, length_includes_head=True, color='r')
plt.gca().set_aspect('equal')
plt.xlim(-1.2, 1.6)
plt.ylim(-1.2, 1.2)
plt.grid(True)
plt.title('V_a, V_b, V_c and V_ab (red, drawn from V_b)')
plt.show()

## Three-phase power is constant in time

Each phase power pulsates at twice the supply frequency; the three pulsations cancel and the total is
$3 V I \cos\varphi$ at every instant.

In [ ]:
t, p_phase, p_total = three_phase.instantaneous_power(230.0, 10.0, 30.0)
for name, p in zip('abc', p_phase, strict=True):
    plt.plot(1000 * t, p, label=f'p_{name}(t)')
plt.plot(1000 * t, p_total, 'k', linewidth=2, label='total')
plt.xlabel('t (ms)')
plt.ylabel('W')
plt.legend()
plt.grid(True)
plt.show()
print(f'total power: min {p_total.min():.1f} W, max {p_total.max():.1f} W')

## Symmetrical components (awareness)

A balanced set is pure positive sequence; a 10 % dip on phase b spreads 0.033 pu into the zero and negative
sequences.

In [ ]:
for v_abc in (three_phase.balanced_set(), np.array([1.0, 0.9 * three_phase.A_OP**2, three_phase.A_OP])):
    print(np.round(np.abs(three_phase.symmetrical_components(v_abc)), 4))